In [1]:
import numpy as np
import pandas as pd

In [2]:
data1 = pd.read_csv(r'E:\DS-ML-NLP\dataset\CICIDS-2017\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')
data2 = pd.read_csv(r'E:\DS-ML-NLP\dataset\CICIDS-2017\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')
data3=pd.read_csv(r"E:\DS-ML-NLP\dataset\CICIDS-2017\Friday-WorkingHours-Morning.pcap_ISCX.csv")
data4=pd.read_csv(r"E:\DS-ML-NLP\dataset\CICIDS-2017\Monday-WorkingHours.pcap_ISCX.csv")
data5=pd.read_csv(r"E:\DS-ML-NLP\dataset\CICIDS-2017\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv")
data6=pd.read_csv(r"E:\DS-ML-NLP\dataset\CICIDS-2017\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv")
data7=pd.read_csv(r"E:\DS-ML-NLP\dataset\CICIDS-2017\Tuesday-WorkingHours.pcap_ISCX.csv")
data8=pd.read_csv(r"E:\DS-ML-NLP\dataset\CICIDS-2017\Wednesday-workingHours.pcap_ISCX.csv")

In [3]:

data_list = [data1, data2, data3, data4, data5, data6, data7, data8]

print('Data dimensions: ')
for i, data in enumerate(data_list, start = 1):
  rows, cols = data.shape
  print(f'Data{i} -> {rows} rows, {cols} columns')

Data dimensions: 
Data1 -> 225745 rows, 79 columns
Data2 -> 286467 rows, 79 columns
Data3 -> 191033 rows, 79 columns
Data4 -> 529918 rows, 79 columns
Data5 -> 288602 rows, 79 columns
Data6 -> 170366 rows, 79 columns
Data7 -> 445909 rows, 79 columns
Data8 -> 692703 rows, 79 columns


In [4]:

data = pd.concat(data_list)
rows, cols = data.shape

print('New dimension:')
print(f'Number of rows: {rows}')
print(f'Number of columns: {cols}')
print(f'Total cells: {rows * cols}')

New dimension:
Number of rows: 2830743
Number of columns: 79
Total cells: 223628697


In [35]:
for d in data_list: del d

In [6]:
dups = data[data.duplicated()]
print(f'Number of duplicates: {len(dups)}')

Number of duplicates: 308381


In [7]:
data.drop_duplicates(inplace = True)
data.shape

(2522362, 79)

In [8]:
data.head(4)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [9]:
data.isnull().sum()

 Destination Port              0
 Flow Duration                 0
 Total Fwd Packets             0
 Total Backward Packets        0
Total Length of Fwd Packets    0
                              ..
Idle Mean                      0
 Idle Std                      0
 Idle Max                      0
 Idle Min                      0
 Label                         0
Length: 79, dtype: int64

In [10]:
# Clean column names first
data.columns = data.columns.str.strip()

# Now try again
med_flow_bytes = data['Flow Bytes/s'].median()
med_flow_packets = data['Flow Packets/s'].median()  # Should work now

print('Median of Flow Bytes/s:', med_flow_bytes)
print('Median of Flow Packets/s:', med_flow_packets)

Median of Flow Bytes/s: 3722.028051
Median of Flow Packets/s: 70.08936394


In [11]:
data['Flow Bytes/s'] = data['Flow Bytes/s'].fillna(med_flow_bytes)
data['Flow Packets/s'] = data['Flow Packets/s'].fillna(med_flow_packets)

In [12]:
data['Label'].unique()

<ArrowStringArray>
[                    'BENIGN',                       'DDoS',
                   'PortScan',                        'Bot',
               'Infiltration',   'Web Attack � Brute Force',
           'Web Attack � XSS', 'Web Attack � Sql Injection',
                'FTP-Patator',                'SSH-Patator',
              'DoS slowloris',           'DoS Slowhttptest',
                   'DoS Hulk',              'DoS GoldenEye',
                 'Heartbleed']
Length: 15, dtype: str

In [13]:
data = data.drop(columns=["Destination Port"])

In [14]:
import pandas as pd

# assuming your dataframe is df and column name is 'label'
data['Label'] = data['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

In [15]:
X = data.drop(columns=['Label'])
y = data['Label']


### outlier

In [31]:
outlier_summary = []

for col in data.drop(columns=['Label']).select_dtypes(include=np.number).columns:

    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    percent = ((data[col] < lower) | (data[col] > upper)).mean() * 100

    outlier_summary.append([col, percent])

outlier_df = pd.DataFrame(outlier_summary, columns=["Feature", "Outlier %"])

print(outlier_df)
del outlier_df

                        Feature  Outlier %
0                 Flow Duration  18.562879
1             Total Fwd Packets  10.030479
2        Total Backward Packets   9.484087
3   Total Length of Fwd Packets  12.501655
4   Total Length of Bwd Packets  22.821308
..                          ...        ...
72                   Active Min  22.154829
73                    Idle Mean  22.488247
74                     Idle Std   9.088426
75                     Idle Max  22.488247
76                     Idle Min  22.488247

[77 rows x 2 columns]


### skewness

In [17]:


print(np.isinf(X).sum().sum())      # total inf values
print(np.isnan(X).sum().sum())     # total NaN values

2775
0


In [18]:
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

In [19]:
X = X.loc[:, X.var() != 0]

In [20]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import PowerTransformer, RobustScaler

In [21]:
# 3. Train-test split (FIRST)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [22]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('power', PowerTransformer(method='yeo-johnson')),
    ('scaler', RobustScaler())
])

X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

In [23]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    max_depth=5,
    learning_rate=0.1,
    n_estimators=200,
    reg_lambda=1,
    reg_alpha=0.5,
    eval_metric='logloss',
    n_jobs=-1
)

xgb.fit(X_train_processed, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [25]:
y_pred = xgb.predict(X_test_processed)

In [28]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9987139048923946

Confusion Matrix:
 [[523691    430]
 [   381 106089]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    524121
           1       1.00      1.00      1.00    106470

    accuracy                           1.00    630591
   macro avg       1.00      1.00      1.00    630591
weighted avg       1.00      1.00      1.00    630591



In [30]:

y_train_pred = xgb.predict(X_train_processed)
train_acc = accuracy_score(y_train, y_train_pred)
y_test_pred = xgb.predict(X_test_processed)
test_acc = accuracy_score(y_test, y_test_pred)
print("Train Accuracy:", train_acc)

print("Test Accuracy:", test_acc)



Train Accuracy: 0.9987773361574948
Test Accuracy: 0.9987139048923946


In [29]:
from sklearn.metrics import roc_auc_score

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

ROC-AUC: 0.9999628869602817
